# Bayesian MMM with Google Meridian

This notebook fits **Google Meridian** (Bayesian marketing mix model; MCMC / NUTS), not a frequentist ridge+OLS pipeline.

**Setup:** Python **3.11–3.13** recommended (`pip install google-meridian`). CPU works for small MCMC settings; use a GPU for larger `n_keep` / many channels.

**Docs:** [Install](https://developers.google.com/meridian/docs/user-guide/installing) · [Getting started Colab](https://developers.google.com/meridian/notebook/meridian-getting-started) · [Configure model](https://developers.google.com/meridian/docs/user-guide/configure-model)

**Inputs (national, revenue KPI):** weekly `revenue`, `time`, constant `population`, and per channel `{channel}_impressions` + `{channel}_spend`. The Streamlit app builds the same layout from the synthetic simulator (`week` → `time`).

**Training:** `sample_prior` then `sample_posterior` (NUTS chains). Tune `n_adapt`, `n_burnin`, `n_keep`, `n_chains` for accuracy vs runtime. Check **R̂** (≈1 is good).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp

from meridian import constants
from meridian.data import data_frame_input_data_builder
from meridian.model import model, prior_distribution, spec

tf.get_logger().setLevel("ERROR")

## 1. Load data

Use the CSV your pipeline writes, or match this schema. Below: Meridians public demo (geo-level); for **only national** rows, filter to one geo or use the Streamlit-generated CSV.

In [ ]:
url = "https://raw.githubusercontent.com/google/meridian/refs/heads/main/meridian/data/simulated_data/csv/geo_all_channels.csv"
df = pd.read_csv(url)
df.head()

## 2. Build `InputData` (same pattern as [Getting Started](https://developers.google.com/meridian/notebook/meridian-getting-started))

In [ ]:
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type="non_revenue",
    default_kpi_column="conversions",
    default_revenue_per_kpi_column="revenue_per_conversion",
)
builder = (
    builder.with_kpi(df)
    .with_revenue_per_kpi(df)
    .with_population(df)
    .with_controls(df, control_cols=["sentiment_score_control", "competitor_sales_control"])
)
channels = ["Channel0", "Channel1", "Channel2", "Channel3", "Channel4"]
builder = builder.with_media(
    df,
    media_cols=[f"{c}_impression" for c in channels],
    media_spend_cols=[f"{c}_spend" for c in channels],
    media_channels=channels,
)
data = builder.build()

## 3. Prior + model + MCMC

In [ ]:
roi_mu, roi_sigma = 0.2, 0.9
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(roi_mu, roi_sigma, name=constants.ROI_M)
)
model_spec = spec.ModelSpec(prior=prior, enable_aks=True)
mmm = model.Meridian(input_data=data, model_spec=model_spec)

mmm.sample_prior(500)
mmm.sample_posterior(n_chains=10, n_adapt=2000, n_burnin=500, n_keep=1000, seed=0)
print("Done.")

## 4. Quick posterior check (ArviZ)

In production, use `meridian.analysis.visualizer` / `Summarizer` for plots and two-pagers (see Getting Started notebook).

In [ ]:
import arviz as az
az.summary(mmm.inference_data).head(20)